In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path(".")

files = [
    ("A_7_5g",  ROOT/"AMhMg_7_5g"/"AMhMg_7_5g_clean.csv"),
    ("A_15g",   ROOT/"AMhMg_15g"/"AMhMg_15g_clean.csv"),
    ("A_18g",   ROOT/"AMhMg_18g"/"AMhMg_18g_clean.csv"),
    ("A_22_5g", ROOT/"AMhMg_22_5g"/"AMhMg_22_5g_clean.csv"),
]

def load(path):
    df = pd.read_csv(path)
    # Arreglar por si quedaron nombres raros:
    df = df.rename(columns={c.lower().strip():c for c in df.columns})
    if not {"nm","A"}.issubset(df.columns):
        # Si venía como x,y entonces lo corregimos
        c0, c1 = df.columns[:2]
        df = df.rename(columns={c0:"nm", c1:"A"})
    df["nm"] = pd.to_numeric(df["nm"], errors="coerce")
    df["A"]  = pd.to_numeric(df["A"], errors="coerce")
    return df.dropna()[["nm","A"]].sort_values("nm").drop_duplicates()

# ---- MATRIZ SIN INTERPOLAR (INTERSECCIÓN EXACTA) ----

mat = load(files[0][1]).rename(columns={"A": files[0][0]})

for label, path in files[1:]:
    df = load(path).rename(columns={"A": label})
    mat = mat.merge(df, on="nm", how="inner")   # SOLO nm que existen en todos

mat = mat.sort_values("nm").reset_index(drop=True)

out = ROOT / "AMhMg_matrix.csv"
mat.to_csv(out, index=False)

print("✔ Matriz generada correctamente:")
print(out)
display(mat.head())

✔ Matriz generada correctamente:
AMhMg_matrix.csv


,nm,A_7_5g,A_15g,A_18g,A_22_5g
0,219.5,0.730,1.679,1.551,1.291
1,220.0,0.713,1.626,1.507,1.256
2,220.5,0.697,1.578,1.465,1.223
3,221.0,0.681,1.536,1.425,1.191
4,221.5,0.665,1.495,1.386,1.160
